In [1]:
import langchain
print(langchain.__version__)

1.2.10


1. create_agent라는 새 API가 등장했습니다. 기존의 복잡한 에이전트 구성 대신 좀 더 간단하면서도 확장 가능한 방식으로 에이전트를 만들 수 있도록 변경되었습니다. 


1. 미들웨어(Middleware) 개념이 강화되어, 요청 → 모델 호출 → 툴 실행 → 결과 처리 등의 흐름 내에 훅(hook)을 넣어 사용자 흐름에 맞게 커스터마이징 가능해졌습니다. 



1. 내부 구조가 리팩토링되어, 핵심 API 네임스페이스가 좀 더 “에이전트 중심”으로 정리되고 불필요한 추상화나 레거시 코드가 정리되었습니다.

https://docs.langchain.com/oss/python/langchain/models#azure

In [2]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
if AZURE_OPENAI_API_KEY:
    print(AZURE_OPENAI_API_KEY[:10])
else:
    print("AZURE_OPENAI_API_KEY가 설정되지 않았습니다.")
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    langsmith_key = os.getenv('LANGSMITH_API_KEY')
    if langsmith_key and len(langsmith_key) > 0:
        print('랭스미스로 추적 중입니다 :', langsmith_key[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano


In [3]:
import os
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version="2025-03-01-preview",
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
llm.invoke("안녕!")

AIMessage(content='안녕! 반가워요. 무엇을 도와드릴까요? 원하시는 주제나 질문을 말해주면 바로 도와드릴게요. 예시:\n- 번역, 요약, 글쓰기/교정\n- 코딩이나 기술 질문\n- 학습 계획이나 아이디어 뇌폭풍\n- 정보 검색이나 비교 분석\n\n어떤 걸 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 481, 'prompt_tokens': 9, 'total_tokens': 490, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DCqMNroj0ykt7oTec5vG3M4QcRWKK', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reaso

In [4]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("너는 한국어를 프랑스어로 번역하는 번역가야."),
    HumanMessage("안녕!"),
    AIMessage("Bonjour!"),
    HumanMessage("오늘은 평소보다 더 졸리네")
]

response = llm.invoke(conversation).content
print(response)

for chunk in llm.stream("프로그래밍이 어려울 때에는 어떻게 해야해? 짧게 정리해줘"):
    print(chunk.text, end="|", flush=True) #<-- 청크 단위가 어떻게 나오는지 확인해봅시다.

Aujourd'hui, je suis plus somnolent(e) que d'habitude.
||-| 문제|를| 명|확|히| 이해|하기|:| 입력|/|출|력|과| 제|약| 조건| 확인|.
|-| 큰| 문제|를| 작은| 단|위|로| 나|누|기|,| 먼저| 의|사|코|드|나| 계획| 작성|.
|-| 작은| 예|제|부터| 차|근|차|근| 구현|하고| 점|진|적으로| 확|장|.
|-| 디|버|깅|은| 체|계|적으로|:| 출력|/log| 남|기|기|,| 단|위| 테스트|로| 원|인| 좁|히|기|.
|-| 에|러| 메시|지| 잘| 읽|고|,| 필요한| 경우| 인터넷| 검색|으로| 해결|하|되| 키|워|드| 정확|히|.
|-| 도움| 받|기|:| 구|체|적으로| 설명|하고| 재|현| 코드|와| 기대| 결과|를| 함께| 공유|.
|-| 짧|게| 쉬|었다|가| 재|도|전|하기|;| 뇌|를| 리|프|레|시|한| 뒤| 다시| 시|도|.||||

## 모델에 tool 바인딩하기

In [5]:
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

llm_with_tools = llm.bind_tools([get_weather])  

response = llm_with_tools.invoke("보스턴의 날씨는?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


### 구조화된 출력 얻기

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="영화 제목")
    year: int = Field(..., description="영화 개봉년도")
    director: str = Field(..., description="영화 감독")
    rating: float = Field(..., description="영화 평점")

model_with_structure = llm.with_structured_output(Movie, method="function_calling")
response = model_with_structure.invoke("인셉션에 대해서 설명해줘.")
print(response)
print(response.title)
print(response.year)
print(response.director)
print(response.rating)

title='인셉션' year=2010 director='크리스토퍼 놀란' rating=8.8
인셉션
2010
크리스토퍼 놀란
8.8
